## Imports  

In [7]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

## Connection 

## Mapping columns to data 

In [8]:
FILE_PATH = r"data/chase/raw/Chase1664_Activity20221126_20221231_20241127.CSV"
BANK_CODE = "Chase"
ACCOUNT_LAST4 = "1664"

## connect + fetch account_id

In [9]:
conn = psycopg2.connect(
    user='postgres',
    password='postgres',
    host='localhost',
    port='5432',
    database='postgres'
)

cur = conn.cursor()

cur.execute("""
             SELECT a.account_id
             FROM accounts a
             JOIN banks b on a.bank_id = b.bank_id  
             WHERE b.bank_code = %s AND a.account_last4 = %s
             """, (BANK_CODE, ACCOUNT_LAST4)          
)

row = cur.fetchone()
if not row:
    raise ValueError("No matching account_id found for bank + last4")
account_id = row[0]
account_id

1

## read CSV + build canonical dataset

In [12]:
df = pd.read_csv(FILE_PATH)
source_filename = os.path.basename(FILE_PATH)

# Build canonical columns
canonical["transaction_date"] = pd.to_datetime(df["Transaction Date"]).dt.date
canonical["account_id"] = account_id  # Better
canonical["transaction_date"] = pd.to_datetime(df["Transaction Date"]).dt.date
canonical["posted_date"] = pd.to_datetime(df["Post Date"], errors="coerce").dt.date
canonical["amount_cents"] = (df["Amount"].astype(float) * 100).round().astype(int)
canonical["description"] = df["Description"].fillna("").astype(str)
canonical["raw_type"] = df["Type"].fillna("").astype(str)
canonical["memo"] = df.get("Memo", pd.Series([""] * len(df))).fillna("").astype(str)

# map raw type to transaction_type
type_map = {
    "Sale": "purchase",
    "Payment": "payment",
    "Fee": "fee",
    "Refund": "refund",
    "Return": "refund",
    "Adjustment": "other"
}
canonical["transaction_type"] = canonical["raw_type"].map(type_map).fillna("other")

# traceability
canonical["source_filename"] = source_filename
canonical["source_row_id"] = df.index + 1  # 1-based row id

# simple duplicate key
normalized_desc = canonical["description"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
canonical["possible_duplicate_key"] = (
    BANK_CODE + "|" + ACCOUNT_LAST4 + "|" +
    canonical["transaction_date"].astype(str) + "|" +
    canonical["amount_cents"].astype(str) + "|" +
    normalized_desc
)

# optional fields not used yet
canonical["category_id"] = None
canonical["merchant_id"] = None
canonical["address"] = None
canonical["reference_number"] = None
canonical["raw_metadata"] = None

canonical.head()


,account_id,transaction_date,posted_date,amount_cents,description,raw_type,memo,transaction_type,source_filename,source_row_id,possible_duplicate_key,category_id,merchant_id,address,reference_number,raw_metadata
0,1,2022-12-27,2022-12-28,-3000,CHEVRON 0208461,Sale,,purchase,Chase1664_Activity20221126_20221231_20241127.CSV,1,Chase|1664|2022-12-27|-3000|CHEVRON 0208461,None,None,None,None,None
1,1,2022-12-25,2022-12-27,-2523,ARCO #42810,Sale,,purchase,Chase1664_Activity20221126_20221231_20241127.CSV,2,Chase|1664|2022-12-25|-2523|ARCO #42810,None,None,None,None,None
2,1,2022-12-23,2022-12-26,-1031,TACONTENTO MEXICAN RESTAU,Sale,,purchase,Chase1664_Activity20221126_20221231_20241127.CSV,3,Chase|1664|2022-12-23|-1031|TACONTENTO MEXICAN...,None,None,None,None,None
3,1,2022-12-25,2022-12-26,-549,CVS CarePass,Sale,,purchase,Chase1664_Activity20221126_20221231_20241127.CSV,4,Chase|1664|2022-12-25|-549|CVS CAREPASS,None,None,None,None,None
4,1,2022-12-23,2022-12-25,-311,WAL-MART #5807,Sale,,purchase,Chase1664_Activity20221126_20221231_20241127.CSV,5,Chase|1664|2022-12-23|-311|WAL-MART #5807,None,None,None,None,None


## insert into transactions

In [19]:
rows = canonical[[
    "account_id","transaction_date","amount_cents","description","transaction_type",
    "source_filename","source_row_id","possible_duplicate_key",
    "posted_date","category_id","merchant_id","raw_type","memo","address","reference_number","raw_metadata"
]].values.tolist()

sql = """
INSERT INTO transactions (
  account_id, transaction_date, amount_cents, description, transaction_type,
  source_filename, source_row_id, possible_duplicate_key,
  posted_date, category_id, merchant_id, raw_type, memo, address, reference_number, raw_metadata
) VALUES %s
"""

execute_values(cur, sql, rows)
conn.commit()


UniqueViolation: duplicate key value violates unique constraint "uq_transactions_source"
DETAIL:  Key (source_filename, source_row_id)=(Chase1664_Activity20221126_20221231_20241127.CSV, 1) already exists.


In [22]:
cur.execute("SELECT COUNT(*) FROM transactions;")
cur.fetchone()


(25,)